# Análises: da prata ao ouro

Este notebook acompanha a construção da camada ouro. Não faz parte do pipeline e não grava tabelas.

- A **parte 1** roda sobre a prata, antes do `Gold_Modelagem`, e verifica as condições que o modelo pressupõe: que os joins não multiplicam linhas, quantos pedidos cada período tem e qual o efeito da regra de escolha da avaliação.
- A **parte 2** roda depois do `Gold_Modelagem` e valida as tabelas criadas: granularidade, cobertura, totais e aplicação correta de cada regra de negócio.

Cada verificação traz o que se espera encontrar. Um resultado diferente do esperado indica erro na construção, e não no dado, porque os dados já foram diagnosticados no notebook `Analises_Bronze-Prata`.

## Parte 1. Preparação: verificações sobre a prata

### 1.1 Os joins não multiplicam linhas

As duas tabelas do ouro juntam várias tabelas da prata. Um join só preserva a granularidade se o lado que entra tiver no máximo uma linha por chave. Se um `customer_id`, `seller_id` ou `product_id` se repetisse, cada item ou pedido seria duplicado sem aviso.

**Esperado:** em cada par, linhas iguais a chaves distintas.

In [0]:
SELECT 'customers' AS tabela, COUNT(*) AS linhas, COUNT(DISTINCT customer_id) AS chaves FROM pospucrio.silver.customers
UNION ALL
SELECT 'sellers',  COUNT(*), COUNT(DISTINCT seller_id)  FROM pospucrio.silver.sellers
UNION ALL
SELECT 'products', COUNT(*), COUNT(DISTINCT product_id) FROM pospucrio.silver.products
UNION ALL
SELECT 'orders',   COUNT(*), COUNT(DISTINCT order_id)   FROM pospucrio.silver.orders

### 1.2 Tamanho de cada trimestre

Conta pedidos, totais e entregues, por trimestre, a partir de 2017. Mostra o tamanho de cada ponto da série trimestral e confirma o recorte: do 1º trimestre de 2017 ao 2º de 2018 completos, 3º de 2018 parcial.

**Esperado:** trimestres de 2017-T1 a 2018-T2 com milhares de pedidos cada, 2018-T3 menor, sem setembro, 2018-T4 praticamente vazio.

In [0]:
SELECT
  year(order_purchase_timestamp)       AS ano,
  quarter(order_purchase_timestamp)    AS trimestre,
  COUNT(*)                             AS pedidos,
  COUNT_IF(order_status = 'delivered') AS entregues
FROM pospucrio.silver.orders
WHERE order_purchase_timestamp >= '2017-01-01'
GROUP BY 1, 2
ORDER BY 1, 2

### 1.3 Efeito da regra de escolha da avaliação

Compara, nos pedidos com mais de uma avaliação, a nota escolhida por duas regras: a avaliação de resposta mais recente (a adotada) e a de criação mais recente (a considerada inicialmente). Mostra em quantos pedidos a escolha da regra muda a nota.

`max_by(valor, criterio)` devolve o valor da linha em que o critério é máximo.

In [0]:
WITH multi AS (
  SELECT
    order_id,
    max_by(review_score, review_answer_timestamp) AS nota_por_resposta,
    max_by(review_score, review_creation_date)    AS nota_por_criacao
  FROM pospucrio.silver.order_reviews
  GROUP BY order_id
  HAVING COUNT(*) > 1
)
SELECT
  COUNT(*)                                          AS pedidos_com_varias_avaliacoes,
  COUNT_IF(nota_por_resposta <> nota_por_criacao)   AS regras_divergem
FROM multi

## Parte 2. Validação: verificações sobre o ouro

### 2.1 gold.pedidos: granularidade e cobertura

Confere que há uma linha por pedido, que nenhum pedido da prata no período se perdeu, e quantos ficaram sem estado, valor ou nota.

**Esperado:**
- `linhas` igual a `pedidos_distintos` e igual a `pedidos_prata`.
- `sem_estado` igual a zero (todo pedido tem cliente).
- `sem_valor` igual a zero: o único pedido sem pagamento é de 2016 e está fora do período.
- `sem_nota` perto de 768, número de pedidos sem avaliação, menos os de 2016.

In [0]:
SELECT
  COUNT(*)                                        AS linhas,
  COUNT(DISTINCT order_id)                        AS pedidos_distintos,
  (SELECT COUNT(*) FROM pospucrio.silver.orders
   WHERE order_purchase_timestamp >= '2017-01-01') AS pedidos_prata,
  COUNT_IF(estado_cliente IS NULL)                AS sem_estado,
  COUNT_IF(valor_pago IS NULL)                    AS sem_valor,
  COUNT_IF(nota IS NULL)                          AS sem_nota
FROM pospucrio.gold.pedidos

### 2.2 gold.pedidos: valor pago

A soma do valor pago no ouro deve ser igual à soma dos pagamentos da prata para os mesmos pedidos. Uma diferença indicaria pagamentos perdidos ou multiplicados na agregação.

**Esperado:** as duas colunas iguais.

In [0]:
SELECT
  (SELECT SUM(valor_pago) FROM pospucrio.gold.pedidos) AS total_ouro,
  (SELECT SUM(pg.payment_value)
   FROM pospucrio.silver.order_payments pg
   JOIN pospucrio.silver.orders o ON pg.order_id = o.order_id
   WHERE o.order_purchase_timestamp >= '2017-01-01')    AS total_prata

### 2.3 gold.pedidos: forma de pagamento

A primeira consulta mostra a distribuição da forma de pagamento. A segunda explica cada pedido com forma nula, pelo motivo.

**Esperado:** "à vista" e "parcelado" com dezenas de milhares de pedidos cada, poucos nulos, todos explicados por `not_defined` ou por parcelas inválidas, e nenhum por falta de pagamento.

In [0]:
SELECT
  coalesce(forma_pagamento, '(nulo)')   AS forma_pagamento,
  COUNT(*)                              AS pedidos,
  COUNT_IF(status = 'delivered')        AS entregues
FROM pospucrio.gold.pedidos
GROUP BY 1
ORDER BY pedidos DESC

In [0]:
WITH pg AS (
  SELECT
    order_id,
    COUNT_IF(payment_type = 'not_defined')                                AS nao_definidos,
    COUNT_IF(payment_type <> 'voucher' AND payment_installments IS NULL)  AS parcelas_invalidas
  FROM pospucrio.silver.order_payments
  GROUP BY order_id
)
SELECT
  CASE WHEN pg.order_id IS NULL        THEN 'sem pagamento'
       WHEN pg.nao_definidos > 0       THEN 'not_defined'
       WHEN pg.parcelas_invalidas > 0  THEN 'parcelas inválidas'
       ELSE 'sem motivo (erro)' END    AS motivo,
  COUNT(*)                             AS pedidos
FROM pospucrio.gold.pedidos p
LEFT JOIN pg ON p.order_id = pg.order_id
WHERE p.forma_pagamento IS NULL
GROUP BY 1

### 2.4 gold.pedidos: método principal

Cruza a combinação de meios usada em cada pedido com o método principal atribuído. Cada combinação deve levar ao método que a regra define.

**Esperado:**
- "credit_card" e "credit_card + voucher" levam a "cartão de crédito".
- "voucher" sozinho leva a "voucher".
- "boleto" e "debit_card" levam a "boleto" e "cartão de débito".
- "credit_card + debit_card" leva ao meio de maior valor.
- "not_defined" leva a nulo.
- Nenhuma combinação aparece com dois métodos diferentes, exceto a de crédito com débito, que depende do valor.

In [0]:
WITH tipos AS (
  SELECT order_id, array_join(array_sort(collect_set(payment_type)), ' + ') AS combinacao
  FROM pospucrio.silver.order_payments
  GROUP BY order_id
)
SELECT
  t.combinacao,
  coalesce(p.metodo_principal, '(nulo)') AS metodo_principal,
  COUNT(*)                               AS pedidos
FROM pospucrio.gold.pedidos p
LEFT JOIN tipos t ON p.order_id = t.order_id
GROUP BY 1, 2
ORDER BY pedidos DESC

### 2.5 gold.pedidos: coerência entre forma e método

Só o cartão de crédito admite parcelamento na origem. Um pedido parcelado com outro método principal indicaria erro na regra.

**Esperado:** "parcelado" aparece apenas com "cartão de crédito".

In [0]:
SELECT
  coalesce(forma_pagamento, '(nulo)')  AS forma_pagamento,
  coalesce(metodo_principal, '(nulo)') AS metodo_principal,
  COUNT(*)                             AS pedidos
FROM pospucrio.gold.pedidos
GROUP BY 1, 2
ORDER BY 1, 2

### 2.6 gold.pedidos: nota escolhida

Compara a nota gravada no ouro com a da avaliação de resposta mais recente, calculada de forma independente. A segunda consulta mostra exemplos de pedidos com avaliações de notas diferentes, para inspeção visual.

**Esperado:** zero divergências.

In [0]:
WITH ultima AS (
  SELECT order_id, max_by(review_score, review_answer_timestamp) AS nota_esperada
  FROM pospucrio.silver.order_reviews
  GROUP BY order_id
)
SELECT COUNT(*) AS divergencias
FROM pospucrio.gold.pedidos p
JOIN ultima u ON p.order_id = u.order_id
WHERE p.nota <> u.nota_esperada

In [0]:
WITH multi AS (
  SELECT order_id
  FROM pospucrio.silver.order_reviews
  GROUP BY order_id
  HAVING COUNT(DISTINCT review_score) > 1
)
SELECT r.order_id, r.review_score, r.review_creation_date, r.review_answer_timestamp,
       p.nota AS nota_escolhida
FROM pospucrio.silver.order_reviews r
JOIN multi m           ON r.order_id = m.order_id
JOIN pospucrio.gold.pedidos p ON r.order_id = p.order_id
ORDER BY r.order_id, r.review_answer_timestamp DESC
LIMIT 30

### 2.7 gold.itens: granularidade, cobertura e totais

Confere uma linha por item, nenhum item perdido no período, nenhum item sem estado e a soma dos preços igual à da prata.

**Esperado:**
- `linhas`, `chaves` e `itens_prata` iguais.
- `sem_estado_cliente` e `sem_estado_vendedor` iguais a zero.
- `sem_categoria` perto de 1.603, menos os itens de 2016.
- `total_preco` igual a `total_preco_prata`.

In [0]:
WITH prata AS (
  SELECT i.price
  FROM pospucrio.silver.order_items i
  JOIN pospucrio.silver.orders o ON i.order_id = o.order_id
  WHERE o.order_purchase_timestamp >= '2017-01-01'
)
SELECT
  COUNT(*)                                   AS linhas,
  COUNT(DISTINCT order_id, order_item_id)    AS chaves,
  (SELECT COUNT(*) FROM prata)               AS itens_prata,
  COUNT_IF(estado_cliente IS NULL)           AS sem_estado_cliente,
  COUNT_IF(estado_vendedor IS NULL)          AS sem_estado_vendedor,
  COUNT_IF(categoria = 'sem categoria')      AS sem_categoria,
  SUM(preco)                                 AS total_preco,
  (SELECT SUM(price) FROM prata)             AS total_preco_prata
FROM pospucrio.gold.itens

### 2.8 Consistência entre as duas tabelas

Todo item deve pertencer a um pedido de `gold.pedidos`. O contrário não vale: pedidos cancelados ou indisponíveis podem não ter itens registrados. A segunda consulta mostra o status dos pedidos sem item.

**Esperado:** zero itens sem pedido. Pedidos sem item concentrados em `canceled` e `unavailable`.

In [0]:
SELECT
  (SELECT COUNT(*) FROM pospucrio.gold.itens i
     LEFT ANTI JOIN pospucrio.gold.pedidos p ON i.order_id = p.order_id) AS itens_sem_pedido,
  (SELECT COUNT(*) FROM pospucrio.gold.pedidos p
     LEFT ANTI JOIN pospucrio.gold.itens i ON p.order_id = i.order_id)   AS pedidos_sem_item

In [0]:
SELECT p.status, COUNT(*) AS pedidos_sem_item
FROM pospucrio.gold.pedidos p
LEFT ANTI JOIN pospucrio.gold.itens i ON p.order_id = i.order_id
GROUP BY p.status
ORDER BY pedidos_sem_item DESC

### 2.9 Catálogo de dados no Unity Catalog

Mostra as descrições registradas em cada tabela e coluna. O resultado serve de evidência do catálogo de dados para o relatório, junto com a tela do Catalog Explorer.

In [0]:
DESCRIBE TABLE EXTENDED pospucrio.gold.pedidos

In [0]:
DESCRIBE TABLE EXTENDED pospucrio.gold.itens

### 2.10 Amostra das tabelas

In [0]:
SELECT * FROM pospucrio.gold.pedidos LIMIT 10

In [0]:
SELECT * FROM pospucrio.gold.itens LIMIT 10